In [1]:
%%capture
!pip install unsloth

In [8]:
import random, json, hashlib

SMELLS = ["Long Parameter List", "Primitive Obsession", "Long Method", "Magic Number Risk"]
SYSTEM = (
    "You predict potential code smells from UML class diagrams written in Mermaid. "
    "Look at the method signatures and attributes. "
    "Possible smells: Long Parameter List (6+ args), Primitive Obsession (raw String/int for email, phone, zip, currency), Long Method (names combining 3+ verbs), Magic Number Risk (raw numbers for taxRate, discount, retry, timeout). "
    "Output only a JSON array of objects with keys 'smell', 'class', 'method' (or 'attribute' if no method), 'reason'. "
    "Output [] if there are no potential code smells."
)

NOUNS = ["Order", "Customer", "Product", "Invoice", "Payment", "User", "Account", "Cart", "Ticket", "Booking"]
ROLES = ["Service", "Repository", "Controller", "Manager"]
VERBS = ["create", "update", "delete", "find", "load", "save", "validate", "send", "calculate", "render", "process", "handle"]

class D:
    def __init__(s, rng):
        s.r = rng; s.cls = {}; s.counter = 0
    def name(s):
        s.counter += 1
        return f"{s.r.choice(NOUNS)}{s.r.choice(ROLES)}_{s.counter}"
    def add(s, attrs=None, ops=None):
        r = s.r; n = s.name()
        a = []
        if attrs is not None: a = attrs
        else:
            for _ in range(r.randint(0, 3)):
                vis = r.choice("+-")
                attr_name = r.choice(["id", "name", "status", "price", "quantity", "date", "description", "total", "age", "weight", "active"])
                a.append(f"{vis}{r.choice(['String', 'int', 'double', 'boolean'])} {attr_name}")
        o = []
        if ops is not None: o = ops
        else:
            for _ in range(r.randint(0, 4)):
                o.append(f"+{r.choice(VERBS)}{r.choice(NOUNS)}()")
        s.cls[n] = {"a": a, "o": o}
        return n
    def render(s):
        lines = ["classDiagram"]
        for n, c in s.cls.items():
            if c["a"] or c["o"]:
                lines.append(f"  class {n} {{")
                for m in c["a"] + c["o"]: lines.append(f"    {m}")
                lines.append("  }")
            else:
                lines.append(f"  class {n}")
        return "\n".join(lines)

# --- CLEAR POSITIVE INJECTORS ---
def inject_long_param(d):
    p_count = d.r.randint(6, 9)
    params = [f"arg{i}: {d.r.choice(['String', 'int', 'double', 'boolean', 'Date'])}" for i in range(p_count)]
    d.add(ops=[f"+{d.r.choice(VERBS)}{d.r.choice(NOUNS)}({', '.join(params)})"])

def inject_primitive(d):
    terms = d.r.sample(["email", "phone", "zipCode", "currency"], d.r.randint(2, 4))
    attrs = [f"+String {term}" for term in terms]
    d.add(attrs=attrs)

def inject_long_method(d):
    v1, v2, v3 = d.r.sample(VERBS, 3)
    n1 = d.r.choice(NOUNS)
    d.add(ops=[f"+{v1}And{v2}And{v3}{n1}()"])

def inject_magic_number(d):
    terms = d.r.sample(["taxRate", "discountPercentage", "maxRetryCount", "timeoutSeconds"], d.r.randint(2, 3))
    attrs = [f"+double {term}" for term in terms]
    d.add(attrs=attrs)

# --- CLEAR HARD NEGATIVES (NEAR MISSES) ---
def inject_near_miss_param(d):
    d.add(ops=["+processOrder(a: String, b: int, c: double, d: boolean)"])

def inject_near_miss_primitive(d):
    d.add(attrs=["+int id", "+String name", "+String description", "+boolean status", "+int age"])

def inject_near_miss_method(d):
    d.add(ops=["+loadAndCacheUser()", "+findAndReplaceText()"])

def inject_near_miss_magic(d):
    d.add(attrs=["+int quantity", "+double weight", "+int age"])

def make_example(rng):
    d = D(rng)
    for _ in range(rng.randint(2, 5)): d.add()
    k = rng.choices([0, 1, 2], [0.2, 0.6, 0.2])[0]
    chosen = rng.sample(SMELLS, k)

    if "Long Parameter List" in chosen: inject_long_param(d)
    if "Primitive Obsession" in chosen: inject_primitive(d)
    if "Long Method" in chosen: inject_long_method(d)
    if "Magic Number Risk" in chosen: inject_magic_number(d)

    if "Long Parameter List" not in chosen and rng.random() < 0.5: inject_near_miss_param(d)
    if "Primitive Obsession" not in chosen and rng.random() < 0.5: inject_near_miss_primitive(d)
    if "Long Method" not in chosen and rng.random() < 0.5: inject_near_miss_method(d)
    if "Magic Number Risk" not in chosen and rng.random() < 0.5: inject_near_miss_magic(d)

    labels = []
    for name, c in d.cls.items():
        for op in c["o"]:
            if op.count(",") >= 5:
                labels.append({"smell": "Long Parameter List", "class": name, "method": op.split("(")[0][1:], "reason": "Method has 6 or more parameters."})
            if op.count("And") >= 2:
                labels.append({"smell": "Long Method", "class": name, "method": op.split("(")[0][1:], "reason": "Method name combines 3 or more verbs."})
        for attr in c["a"]:
            if any(term in attr.lower() for term in ["email", "phone", "zip", "currency"]):
                labels.append({"smell": "Primitive Obsession", "class": name, "attribute": attr.split()[-1], "reason": "Domain concept represented as a raw String."})
            if any(term in attr.lower() for term in ["taxrate", "discount", "retry", "timeout"]):
                labels.append({"smell": "Magic Number Risk", "class": name, "attribute": attr.split()[-1], "reason": "Business logic values should be dedicated types, not raw numbers."})

    unique_labels = []
    for l in labels:
        if l not in unique_labels: unique_labels.append(l)
    return d.render(), unique_labels

def build(n_train=4000, n_val=100, n_test=200, seed=42):
    rng = random.Random(seed); seen = set(); data_ = []
    need = n_train + n_val + n_test
    while len(data_) < need:
        ex = make_example(rng)
        if not ex: continue
        h = hashlib.md5(ex[0].encode()).hexdigest()
        if h in seen: continue
        seen.add(h); data_.append(ex)
    def fmt(ex):
        return {"messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": ex[0]}, {"role": "assistant", "content": json.dumps(ex[1])}]}
    rows = [fmt(e) for e in data_]
    return rows[:n_train], rows[n_train:n_train+n_val], rows[n_train+n_val:]

In [9]:
import json
train_rows, val_rows, test_rows = build(n_train=4000, n_val=100, n_test=200)
for name, rows in [("code_smell_train", train_rows), ("code_smell_val", val_rows), ("code_smell_test", test_rows)]:
    with open(f"{name}.jsonl", "w") as f:
        for r in rows: f.write(json.dumps(r) + "\n")

In [4]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit", max_seq_length = max_seq_length, dtype = None, load_in_4bit = True)
model = FastLanguageModel.get_peft_model(
    model, r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth", random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.2: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.11.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


In [5]:
from datasets import Dataset
def to_text(rows):
    return Dataset.from_list([{"text": tokenizer.apply_chat_template(r["messages"], tokenize=False)} for r in rows])
train_ds = to_text(train_rows)

In [6]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

# Increased epochs to 3 and lowered LR to 1e-4 for better convergence
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = train_ds,
    dataset_text_field = "text", max_seq_length = max_seq_length, packing = False,
    args = SFTConfig(per_device_train_batch_size = 2, gradient_accumulation_steps = 4, num_train_epochs = 3, learning_rate = 1e-4, warmup_steps = 10, lr_scheduler_type = "linear", optim = "adamw_8bit", weight_decay = 0.01, fp16 = not is_bfloat16_supported(), bf16 = is_bfloat16_supported(), logging_steps = 10, save_strategy = "no", report_to = "none", seed = 3407, output_dir = "outputs")
)
trainer = train_on_responses_only(trainer, instruction_part = "<|im_start|>user\n", response_part = "<|im_start|>assistant\n")
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/4000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,000 | Num Epochs = 3 | Total steps = 1,500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,0.425365
20,0.195575
30,0.070869
40,0.029507
50,0.036147
60,0.016651
70,0.012746
80,0.007024
90,0.001860
100,0.002782


TrainOutput(global_step=1500, training_loss=0.00556499500713835, metrics={'train_runtime': 3526.1819, 'train_samples_per_second': 3.403, 'train_steps_per_second': 0.425, 'total_flos': 3.778342746349363e+16, 'train_loss': 0.00556499500713835, 'epoch': 3.0})

In [10]:
# Evaluate on held-out test set
import json, torch
from collections import defaultdict
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"

def predict(users, bs=8, max_new=300):
    order = sorted(range(len(users)), key=lambda i: len(users[i])); res = [None]*len(users)
    for s in range(0, len(order), bs):
        idx = order[s:s+bs]
        prompts = [tokenizer.apply_chat_template(
            [{"role":"system","content":SYSTEM},{"role":"user","content":users[i]}],
            tokenize=False, add_generation_prompt=True) for i in idx]
        enc = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, use_cache=True)
        for i, o in zip(idx, tokenizer.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)):
            res[i] = o.strip()
    return res

def parse(s):
    try:
        v = json.loads(s)
        parsed = []
        for x in v:
            loc = x.get("method", x.get("attribute", ""))
            parsed.append((x["smell"], x["class"], loc))
        return parsed, True
    except Exception:
        return [], False

users = [r["messages"][1]["content"] for r in test_rows]
gold  = [parse(r["messages"][2]["content"])[0] for r in test_rows]
preds = predict(users)

tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int)
exact = invalid = 0; clean_ok = clean_tot = 0
for p_raw, g in zip(preds, gold):
    p, ok = parse(p_raw); invalid += (not ok)
    gk = set(g); pk = set(p)
    exact += (gk == pk)
    if not g: clean_tot += 1; clean_ok += (not p)
    for k in pk:
        if k in gk: tp[k[0]] += 1
        else: fp[k[0]] += 1
    for k in gk:
        if k not in pk: fn[k[0]] += 1

print(f"Invalid JSON: {invalid}/{len(preds)}")
print(f"Exact match: {exact/len(preds):.1%}")
print(f"Clean accuracy: {clean_ok}/{clean_tot}")
print(f"\n{'Smell':26s} {'P':>6s} {'R':>6s} {'F1':>6s}")
for s in SMELLS:
    P = tp[s]/max(tp[s]+fp[s],1); R = tp[s]/max(tp[s]+fn[s],1); F = 2*P*R/max(P+R,1e-9)
    print(f"{s:26s} {P:6.2f} {R:6.2f} {F:6.2f}")

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Invalid JSON: 0/200
Exact match: 100.0%
Clean accuracy: 40/40

Smell                           P      R     F1
Long Parameter List          1.00   1.00   1.00
Primitive Obsession          1.00   1.00   1.00
Long Method                  1.00   1.00   1.00
Magic Number Risk            1.00   1.00   1.00


In [11]:
model.save_pretrained("code_smell_lora"); tokenizer.save_pretrained("code_smell_lora")
!zip -r code_smell_lora.zip code_smell_lora
from google.colab import files
files.download("code_smell_lora.zip")

Unsloth: Restored added_tokens_decoder metadata in code_smell_lora/tokenizer_config.json.


  adding: code_smell_lora/ (stored 0%)
  adding: code_smell_lora/README.md (deflated 65%)
  adding: code_smell_lora/adapter_model.safetensors (deflated 8%)
  adding: code_smell_lora/tokenizer_config.json (deflated 89%)
  adding: code_smell_lora/tokenizer.json (deflated 81%)
  adding: code_smell_lora/chat_template.jinja (deflated 71%)
  adding: code_smell_lora/adapter_config.json (deflated 60%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>